In [ ]:
# 1. Imports y rutas
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "tables" / "chapter5"
FIGURES_DIR = PROJECT_ROOT / "figures" / "chapter5"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 2. Cargar data/processed/dataset_cap4_regimes_full.csv en df_full
path = PROCESSED_DATA_DIR / "dataset_cap4_regimes_full.csv"
df_full = pd.read_csv(path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)

print("df_full shape:", df_full.shape)


In [ ]:
# 3. Validar columnas date, sample_split_main, ret_1d, regime
required_cols = ["date", "sample_split_main", "ret_1d", "regime"]
missing_cols = [c for c in required_cols if c not in df_full.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas obligatorias en df_full: {missing_cols}")


In [ ]:
# 4. Validar que df_full contiene train y test
splits_presentes = set(df_full["sample_split_main"].dropna().unique())
required_splits = {"train", "test"}
if not required_splits.issubset(splits_presentes):
    raise ValueError(
        f"sample_split_main no contiene ambos splits requeridos. Presentes: {sorted(splits_presentes)}"
    )


In [ ]:
# 5. Crear df_test filtrando sample_split_main == "test"
df_test = df_full[df_full["sample_split_main"] == "test"].copy()
df_test = df_test.sort_values("date").reset_index(drop=True)

print("df_test shape:", df_test.shape)


In [ ]:
# 6. Validar que df_test no está vacío
if df_test.empty:
    raise ValueError("df_test está vacío tras filtrar sample_split_main == 'test'.")


In [ ]:
# 7. Definir regime_exposure_map
regime_exposure_map = {
    0: 1.0,
    2: 0.75,
    1: 0.25,
}


In [ ]:
# 8. Crear target_exposure con map
# 9. Validar que no quedan target_exposure nulos
df_test["target_exposure"] = df_test["regime"].map(regime_exposure_map)

if df_test["target_exposure"].isna().any():
    faltantes = sorted(df_test.loc[df_test["target_exposure"].isna(), "regime"].dropna().unique())
    raise ValueError(f"Hay regímenes sin mapping en target_exposure: {faltantes}")


In [ ]:
# 10. Crear strategy_exposure = target_exposure.shift(1)
# 11. Crear df_test_bt eliminando strategy_exposure nulo
# 12. Validar que df_test_bt no está vacío
df_test["strategy_exposure"] = df_test["target_exposure"].shift(1)
df_test_bt = df_test.dropna(subset=["strategy_exposure"]).copy()

print("df_test_bt shape:", df_test_bt.shape)

if df_test_bt.empty:
    raise ValueError("df_test_bt está vacío tras eliminar strategy_exposure nulo.")


In [ ]:
# 13. Calcular benchmark_return, strategy_return, benchmark_cum y strategy_cum
df_test_bt["benchmark_return"] = df_test_bt["ret_1d"]
df_test_bt["strategy_return"] = df_test_bt["strategy_exposure"] * df_test_bt["ret_1d"]

df_test_bt["benchmark_cum"] = np.exp(df_test_bt["benchmark_return"].cumsum())
df_test_bt["strategy_cum"] = np.exp(df_test_bt["strategy_return"].cumsum())


In [ ]:
# 14. Calcular metrics_test con total_return, mean_daily_return, daily_volatility, VaR_95, CVaR_95 y max_drawdown
def compute_drawdown(cumulative_returns):
    running_max = cumulative_returns.cummax()
    return cumulative_returns / running_max - 1


def historical_var(returns, alpha=0.95):
    return returns.quantile(1 - alpha)


def historical_cvar(returns, alpha=0.95):
    var = historical_var(returns, alpha)
    return returns[returns <= var].mean()


benchmark_drawdown = compute_drawdown(df_test_bt["benchmark_cum"])
strategy_drawdown = compute_drawdown(df_test_bt["strategy_cum"])

metrics_test = pd.DataFrame(
    {
        "strategy": ["Buy & Hold", "Regime-based"],
        "total_return": [
            df_test_bt["benchmark_cum"].iloc[-1] - 1,
            df_test_bt["strategy_cum"].iloc[-1] - 1,
        ],
        "mean_daily_return": [
            df_test_bt["benchmark_return"].mean(),
            df_test_bt["strategy_return"].mean(),
        ],
        "daily_volatility": [
            df_test_bt["benchmark_return"].std(),
            df_test_bt["strategy_return"].std(),
        ],
        "VaR_95": [
            historical_var(df_test_bt["benchmark_return"]),
            historical_var(df_test_bt["strategy_return"]),
        ],
        "CVaR_95": [
            historical_cvar(df_test_bt["benchmark_return"]),
            historical_cvar(df_test_bt["strategy_return"]),
        ],
        "max_drawdown": [
            benchmark_drawdown.min(),
            strategy_drawdown.min(),
        ],
    }
)

metrics_test


In [ ]:
# 15. Generar gráficos OOS de acumulado y drawdown
plt.figure(figsize=(12, 5))
plt.plot(df_test_bt["date"], df_test_bt["benchmark_cum"], label="Buy & Hold")
plt.plot(df_test_bt["date"], df_test_bt["strategy_cum"], label="Regime-based")
plt.title("Rentabilidad acumulada (Out-of-Sample)")
plt.xlabel("Fecha")
plt.ylabel("Crecimiento acumulado")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df_test_bt["date"], benchmark_drawdown, label="Buy & Hold")
plt.plot(df_test_bt["date"], strategy_drawdown, label="Regime-based")
plt.title("Drawdown (Out-of-Sample)")
plt.xlabel("Fecha")
plt.ylabel("Drawdown")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 16. Exportar:
#     data/processed/backtest_oos_results.csv
#     tables/chapter5/oos_performance_metrics.csv
df_test_bt.to_csv(PROCESSED_DATA_DIR / "backtest_oos_results.csv", index=False)
metrics_test.to_csv(TABLES_DIR / "oos_performance_metrics.csv", index=False)

print("Exportado:", PROCESSED_DATA_DIR / "backtest_oos_results.csv")
print("Exportado:", TABLES_DIR / "oos_performance_metrics.csv")
